# Trabalho Prático: PPO (Proximal Policy Optimization) no Overcooked
# Aprendizado por Reforço 2025/1

#### Thaís Ferreira da Silva - 2021092571

**Última atualização:** Novembro 2025

Este notebook implementa um agente **PPO (Proximal Policy Optimization)** usando **CleanRL** para o ambiente Overcooked customizado.

**Objetivo:** Treinar um agente único para coletar ingredientes, processá-los e entregar receitas no `simple_map`.

**Por que CleanRL PPO?**
- ✅ **Simples e educacional** - Código limpo e fácil de entender
- ✅ **PPO é mais estável** - Melhor que DQN para ambientes complexos
- ✅ **Python 3.13 compatível** - Não depende de bibliotecas antigas
- ✅ **[Tutorial oficial PettingZoo](https://pettingzoo.farama.org/tutorials/cleanrl/)** - Implementação de referência

**Diferenças do ambiente Lunar Lander:**
- Observações: Grid 3D (altura × largura × 8 canais) ao invés de vetor 1D
- Policy: CNN (Convolutional Neural Network) ao invés de MLP
- Ambiente: Multi-agente PettingZoo (usando 1 agente)
- Algoritmo: PPO (policy gradient) ao invés de DQN (value-based)
- **Grid Search:** Busca automática de hiperparâmetros com ranking dos top 10 modelos


## Instalação de Dependências

### **PROBLEMA RESOLVIDO! PyTorch CUDA instalado com sucesso**

**Status da sua instalação:**
- **PyTorch 2.7.1+cu118** (CUDA) instalado na venv
- **NVIDIA GeForce MX350** detectada e funcionando
- **CUDA disponível:** True

### **Comandos de Instalação Completos**

```bash
# 1. Ativar ambiente virtual
.\venv\Scripts\Activate.ps1

# 2. Instalar PyTorch CUDA
pip install torch --index-url https://download.pytorch.org/whl/cu118

# 3. Instalar outras dependências
pip install numpy matplotlib tqdm gymnasium pettingzoo pygame

# 4. Instalar o ambiente Overcooked
pip install -e .
```

### 📋 **O que cada biblioteca faz:**

| **Biblioteca** | **Função** | **Versão** |
|-------------------|---------------|----------------|
| `torch` | Deep learning (PPO, CNN) | **2.7.1+cu118** ✅ |
| `numpy` | Matemática e arrays | 1.21+ |
| `matplotlib` | Gráficos e visualização | 3.5+ |
| `tqdm` | Barras de progresso | Latest |
| `gymnasium` | Interface RL padrão | 0.29+ |
| `pettingzoo` | Multi-agent framework | 1.24+ |
| `pygame` | Renderização visual | 2.5+ |

## 1. Imports e Dependências


In [ ]:
# Bibliotecas padrão
import os
import time
import random
from typing import Dict, Any, Optional, Tuple
from collections import deque

# Bibliotecas de terceiros
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# PyTorch para deep learning
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

# DirectML para AMD GPUs (opcional)
try:
    import torch_directml
    DIRECTML_AVAILABLE = True
except ImportError:
    DIRECTML_AVAILABLE = False

# Gymnasium para interface RL
import gymnasium as gym
from gymnasium import spaces

# Nosso ambiente Overcooked (PettingZoo)
from overcooked_marl_env import OvercookedMARL

print(f"✅ PyTorch {torch.__version__} importado com sucesso!")
print(f"✅ CUDA disponível: {torch.cuda.is_available()}")

# Função para detectar device corretamente
def get_best_device():
    """
    Detecta automaticamente o melhor device disponível.
    
    Prioridade:
    1. CUDA (NVIDIA GPU)
    2. DirectML (AMD GPU)
    3. CPU
    
    Returns:
        tuple: (device, device_name)
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        print(f"NVIDIA GPU detectada: {gpu_name}")
        print("   -> Usando CUDA (recomendado)")
        return device, f"CUDA ({gpu_name})"
    
    if "cuda" not in torch.__version__ and torch.__version__.endswith("+cpu"):
        device = torch.device("cpu")
        return device, "CPU (PyTorch CPU version)"
    
    if DIRECTML_AVAILABLE:
        print("DirectML disponível")
        print("   -> Detectado como AMD GPU")
        print("   -> Usando DirectML")
        device = torch_directml.device()
        return device, "DirectML (AMD GPU)"
    
    # Fallback: CPU
    device = torch.device("cpu")
    print("💻 Usando CPU")
    print("   -> GPU não disponível ou não compatível")
    return device, "CPU"

# Função de teste corrigida
def test_device_compatibility(device):
    """
    Testa se o device suporta as operações necessárias para PPO.
    
    Args:
        device: Device PyTorch para testar
        
    Returns:
        bool: True se compatível, False caso contrário
    """
    try:
        x = torch.randn(4, 5, 7, 8).to(device)
        x_cnn = x.permute(0, 3, 1, 2)
        conv = nn.Conv2d(8, 16, 3, padding=1).to(device)
        y = conv(x_cnn)
        linear = nn.Linear(16 * 5 * 7, 64).to(device)
        z = linear(y.reshape(4, -1))
        loss = z.sum()
        loss.backward()
        return True
    except Exception:
        return False

device, device_name = get_best_device()
is_compatible = test_device_compatibility(device)

if not is_compatible and device.type != "cpu":
    device = torch.device("cpu")
    device_name = "CPU (fallback)"

print(f"Device: {device_name}")


## 2. Wrapper PettingZoo → Gymnasium

CleanRL trabalha com Gymnasium, então precisamos converter o ambiente PettingZoo:


In [ ]:
class PettingZooToGymWrapper(gym.Env):
    """
    Converte ambiente PettingZoo com 1 agente para interface Gymnasium.
    
    CleanRL espera interface Gymnasium padrão (single-agent).
    """
    
    def __init__(self, pettingzoo_env):
        super().__init__()
        self.env = pettingzoo_env
        
        # Verifica se há apenas 1 agente
        if len(self.env.possible_agents) != 1:
            raise ValueError(f"Wrapper funciona apenas com 1 agente. Encontrados: {len(self.env.possible_agents)}")
        
        self.agent_id = self.env.possible_agents[0]
        
        # Define espaços
        self.action_space = self.env.action_spaces[self.agent_id]
        self.observation_space = self.env.observation_spaces[self.agent_id]
        
    def reset(self, seed=None, options=None):
        observations, infos = self.env.reset(seed=seed, options=options)
        return observations[self.agent_id], infos[self.agent_id]
    
    def step(self, action):
        actions = {self.agent_id: action}
        observations, rewards, terminations, truncations, infos = self.env.step(actions)
        
        obs = observations[self.agent_id]
        reward = rewards[self.agent_id]
        terminated = terminations[self.agent_id]
        truncated = truncations[self.agent_id]
        info = infos[self.agent_id]
        
        return obs, reward, terminated, truncated, info
    
    def render(self):
        return self.env.render()
    
    def close(self):
        self.env.close()


## 3. Actor-Critic Network (PPO)

PPO usa uma rede Actor-Critic com CNN para processar observações espaciais:


In [ ]:
class ActorCritic(nn.Module):
    """
    Rede Actor-Critic para PPO com CNN para observações espaciais.
    
    - Actor: Gera distribuição de probabilidades sobre ações
    - Critic: Estima valor do estado (V(s))
    """
    
    def __init__(self, observation_space, action_space):
        super().__init__()
        
        h, w, c = observation_space.shape
        n_actions = action_space.n
        
        # Feature extractor (CNN) compartilhado
        self.cnn = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.Flatten(),
        )
        
        # Calcula tamanho da saída da CNN
        with torch.no_grad():
            sample = torch.zeros(1, c, h, w)
            n_flatten = self.cnn(sample).shape[1]
        
        # Camadas compartilhadas
        self.shared = nn.Sequential(
            nn.Linear(n_flatten, 128),
            nn.ReLU(),
        )
        
        # Actor head (policy)
        self.actor = nn.Linear(128, n_actions)
        
        # Critic head (value function)
        self.critic = nn.Linear(128, 1)
    
    def forward(self, x):
        # x: (batch, height, width, channels) -> (batch, channels, height, width)
        x = x.permute(0, 3, 1, 2)
        
        # Features
        features = self.shared(self.cnn(x))
        
        # Actor: logits das ações
        logits = self.actor(features)
        
        # Critic: valor do estado
        value = self.critic(features)
        
        return logits, value
    
    def get_action_and_value(self, x, action=None):
        """
        Retorna ação, log_prob, entropia e valor.
        Usado durante coleta de dados.
        """
        logits, value = self.forward(x)
        probs = Categorical(logits=logits)
        
        if action is None:
            action = probs.sample()
        
        return action, probs.log_prob(action), probs.entropy(), value


## 4. Inicialização do Ambiente e Configuração PPO


In [ ]:
def generate_hyperparameter_combinations(param_grid: dict) -> list:
    """
    Gera todas as combinações possíveis de hiperparâmetros a partir de um dicionário.
    Args:
        param_grid: dicionário que mapeia nomes dos hiperparâmetros em listas de valores possíveis
    """
    from itertools import product
    
    keys = param_grid.keys()
    values = [param_grid[key] for key in keys]
    combinations_raw = product(*values)
    combinations = [dict(zip(keys, v)) for v in combinations_raw]
    return combinations

In [ ]:
# Grid de hiperparâmetros para busca - VERSÃO MELHORADA
# 🎯 CONFIGURAÇÃO OTIMIZADA: Explora múltiplas estratégias de aprendizado
param_grid = {
    'learning_rate': [1e-3],     # 2 taxas
    'num_steps': [512],           # 3 tamanhos de rollout
    'update_epochs': [4, 8, 16],            # 3 níveis de otimização
    'ent_coef': [0.05, 0.1, 0.15],          # 3 níveis de exploração - CRÍTICO para exploração!
    'clip_coef': [0.1],                 # 2 valores de clipping
    'num_minibatches': [4],           # 3 tamanhos de minibatch
    'gamma': [0.995],                  # 2 horizontes temporais
    'gae_lambda': [0.95, 0.97],              # 2 estimadores de vantagem
    'vf_coef': [0.5],                        # Fixo (balanceado)
    'max_grad_norm': [0.5],                  # Fixo (estável)
    'total_timesteps': [60000],             # Timesteps balanceados
    'seed': [42],                            # Fixo para comparação justa
    
    # NOVO: Parâmetros de exploração forçada
    'exploration_epsilon_start': [0.5],      # Começa com 30% de ações aleatórias
    'exploration_epsilon_end': [0.0],        # Termina sem exploração forçada
    'exploration_decay_steps': [60000]       # Decai ao longo de 30k steps (60% do treino)
}

# param_grid = {
#     'learning_rate': [1e-3],     # 2 taxas
#     'num_steps': [512, 1024],           # 3 tamanhos de rollout
#     'update_epochs': [4, 10, 20],            # 3 níveis de otimização
#     'ent_coef': [0.01, 0.05, 0.1],          # 3 níveis de exploração - CRÍTICO para exploração!
#     'clip_coef': [0.1, 0.2],                 # 2 valores de clipping
#     'num_minibatches': [4, 8, 16],           # 3 tamanhos de minibatch
#     'gamma': [0.99, 0.995],                  # 2 horizontes temporais
#     'gae_lambda': [0.95, 0.97],              # 2 estimadores de vantagem
#     'vf_coef': [0.5],                        # Fixo (balanceado)
#     'max_grad_norm': [0.5],                  # Fixo (estável)
#     'total_timesteps': [50000],             # Timesteps balanceados
#     'seed': [42],                            # Fixo para comparação justa
    
#     # NOVO: Parâmetros de exploração forçada
#     'exploration_epsilon_start': [0.5],      # Começa com 30% de ações aleatórias
#     'exploration_epsilon_end': [0.0],        # Termina sem exploração forçada
#     'exploration_decay_steps': [30000]       # Decai ao longo de 30k steps (60% do treino)
# }

# Configurações fixas do ambiente e modelo
ENVIRONMENT_CONFIG = {
    'map_path': "overcooked_marl_env/map/maps/simple_map.txt",
    'num_fixed_agents': 1,
    'render_mode': None,  # Desabilitado para treinamento rápido
    'verbose': False,
    'max_episode_steps': 800,
    'environment_type': 'Overcooked_MARL'
}

MODEL_CONFIG = {
    'network_type': 'CNN_ActorCritic',
    'cnn_channels': [32, 64, 64],
    'cnn_kernel_sizes': [3, 3, 3],
    'cnn_strides': [1, 1, 1],
    'cnn_paddings': [1, 1, 0],
    'shared_hidden_size': 128,
    'activation_function': 'ReLU'
}

# Gera todas as combinações
combinations = generate_hyperparameter_combinations(param_grid)
print(f"🎯 Total de combinações: {len(combinations)}")

if len(combinations) > 100:
    print(f"⚠️  AVISO: {len(combinations)} combinações podem levar MUITO tempo!")
    print(f"   Estimativa: ~{len(combinations) * 2:.0f} minutos ({len(combinations) * 2 / 60:.1f} horas)")
    print(f"   Considere usar a versão rápida do param_grid (comentada acima)")
else:
    print(f"✅ Quantidade razoável de combinações")
    print(f"   Estimativa: ~{len(combinations) * 2:.0f} minutos")

print(f"\n🔍 NOVIDADE: Exploração forçada ativada!")
print(f"   • Epsilon inicial: {param_grid['exploration_epsilon_start'][0]:.1%}")
print(f"   • Epsilon final: {param_grid['exploration_epsilon_end'][0]:.1%}")
print(f"   • Decaimento ao longo de: {param_grid['exploration_decay_steps'][0]:,} steps")

In [ ]:
# Cria o ambiente base (será recriado para cada configuração)
base_env = OvercookedMARL(
    render_mode=ENVIRONMENT_CONFIG['render_mode'],
    map_path=ENVIRONMENT_CONFIG['map_path'],
    num_fixed_agents=ENVIRONMENT_CONFIG['num_fixed_agents'],
    verbose=ENVIRONMENT_CONFIG['verbose']
)
env = PettingZooToGymWrapper(base_env)

MODEL_CONFIG['observation_space_shape'] = env.observation_space.shape
MODEL_CONFIG['action_space_size'] = env.action_space.n

env.close()
print(f"Ambiente: {env.observation_space.shape} obs, {env.action_space.n} ações")

In [ ]:
def sanity_check_environment():
    """
    Valida se o ambiente está gerando recompensas corretamente.
    Detecta problemas antes de iniciar treinamento longo.
    """
    print("="*70)
    print("🧪 SANITY CHECK: Validando Ambiente")
    print("="*70)
    
    # Cria ambiente de teste
    test_env = OvercookedMARL(
        render_mode=None,
        map_path=ENVIRONMENT_CONFIG['map_path'],
        num_fixed_agents=1,
        verbose=True  # Ativar logs para ver interações
    )
    test_env = PettingZooToGymWrapper(test_env)
    
    obs, _ = test_env.reset()
    rewards_distribution = []
    actions_taken = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}  # Conta cada ação
    
    print(f"\n📊 Testando 200 ações aleatórias...\n")
    
    # Testar 200 ações aleatórias
    for step in range(200):
        action = test_env.action_space.sample()
        actions_taken[action] += 1
        
        obs, reward, done, truncated, info = test_env.step(action)
        
        if reward != 0:
            rewards_distribution.append(reward)
            if len(rewards_distribution) <= 5:  # Mostra primeiras 5
                print(f"   Step {step:3d} | Ação {action} | Reward: {reward:6.2f}")
        
        if done or truncated:
            obs, _ = test_env.reset()
    
    test_env.close()
    
    # Análise dos resultados
    print("\n" + "="*70)
    print("📈 RESULTADOS DO SANITY CHECK")
    print("="*70)
    
    if len(rewards_distribution) == 0:
        print("❌ PROBLEMA CRÍTICO: Nenhuma recompensa não-zero!")
        print("   → O agente NUNCA recebe feedback!")
        print("   → Verifique o sistema de recompensas em interaction_handler.py")
        return False
    
    print(f"✅ Recompensas obtidas: {len(rewards_distribution)}")
    print(f"\n📊 Estatísticas de Recompensas:")
    print(f"   Total de recompensas não-zero: {len(rewards_distribution)}")
    print(f"   Recompensa média: {np.mean(rewards_distribution):.2f}")
    print(f"   Recompensa mínima: {min(rewards_distribution):.2f}")
    print(f"   Recompensa máxima: {max(rewards_distribution):.2f}")
    print(f"   Recompensas únicas: {sorted(set(rewards_distribution))}")
    
    print(f"\n🎮 Distribuição de Ações:")
    total_actions = sum(actions_taken.values())
    action_names = ['↑ Up', '↓ Down', '← Left', '→ Right', '🔧 Interact']
    for action, count in actions_taken.items():
        percentage = (count / total_actions) * 100
        print(f"   {action_names[action]:12s}: {count:3d} ({percentage:5.1f}%)")
    
    # Warnings
    if len(set(rewards_distribution)) == 1:
        print(f"\n⚠️  AVISO: Apenas 1 tipo de recompensa ({rewards_distribution[0]})")
        print("   → Sistema de recompensas pode estar muito simples")
    
    if max(rewards_distribution) > 100:
        print(f"\n⚠️  AVISO: Recompensa muito alta ({max(rewards_distribution)})")
        print("   → Pode causar instabilidade no treinamento")
    
    if actions_taken[4] == 0:
        print(f"\n⚠️  AVISO: Ação 'Interact' nunca foi executada!")
    
    print("\n✅ Sanity check concluído. Ambiente está funcional.\n")
    return True

# Executar sanity check
if not sanity_check_environment():
    print("\n❌ ABORTAR: Corrija os problemas antes de treinar!")
    raise RuntimeError("Ambiente não está funcionando corretamente")

## 🧪 Sanity Checks - Validação do Ambiente

Antes de treinar, vamos garantir que o ambiente está funcionando corretamente:

In [ ]:
# 🔍 TESTE: Verificar se todos os 8 canais são processados pela CNN

print("="*70)
print("TESTE: PROCESSAMENTO DE TODOS OS CANAIS")
print("="*70)

# 1. Criar ambiente de teste
test_env = OvercookedMARL(
    render_mode=None,
    map_path=ENVIRONMENT_CONFIG['map_path'],
    num_fixed_agents=1,
    verbose=False
)
test_wrapper = PettingZooToGymWrapper(test_env)

# 2. Obter uma observação
obs, _ = test_wrapper.reset()
print(f"\n✅ Observação obtida: shape = {obs.shape}")
print(f"   Dimensões: (altura={obs.shape[0]}, largura={obs.shape[1]}, canais={obs.shape[2]})")

# 3. Criar agente de teste
test_agent = ActorCritic(test_wrapper.observation_space, test_wrapper.action_space).to(device)
print(f"\n✅ Rede criada com {obs.shape[2]} canais de entrada")

# 4. Testar processamento
obs_tensor = torch.Tensor(obs).unsqueeze(0).to(device)  # (1, 7, 7, 8)
print(f"\n📊 Input para rede: {obs_tensor.shape}")

# 5. Ver transformação
obs_permuted = obs_tensor.permute(0, 3, 1, 2)  # (1, 8, 7, 7)
print(f"   Após permute: {obs_permuted.shape} (batch, canais, altura, largura)")

# 6. Verificar primeira camada Conv2d
first_conv = test_agent.cnn[0]  # nn.Conv2d(c, 32, ...)
print(f"\n🔬 Primeira camada Conv2d:")
print(f"   Input channels: {first_conv.in_channels}")
print(f"   Output channels: {first_conv.out_channels}")
print(f"   Kernel size: {first_conv.kernel_size}")

# 7. Processar observação completa
with torch.no_grad():
    action, log_prob, entropy, value = test_agent.get_action_and_value(obs_tensor)
    
print(f"\n✅ Processamento concluído com sucesso!")
print(f"   Ação escolhida: {action.item()}")
print(f"   Valor estimado: {value.item():.4f}")

# 8. Testar se TODOS os canais estão sendo usados
print(f"\n🧪 TESTE: Verificar importância de cada canal")

# Criar observações mascaradas (zerando 1 canal por vez)
original_obs = obs_tensor.clone()
canal_impacts = []

for canal_idx in range(8):
    # Zerar um canal
    masked_obs = obs_tensor.clone()
    masked_obs[0, :, :, canal_idx] = 0
    
    # Processar
    with torch.no_grad():
        action_masked, _, _, value_masked = test_agent.get_action_and_value(masked_obs)
    
    # Medir impacto (diferença de valor)
    impact = abs(value.item() - value_masked.item())
    canal_impacts.append((canal_idx, impact))
    
print(f"\n📈 Impacto de cada canal (diferença no valor estimado ao zerar):")
canal_names = [
    "Canal 0: Mapa Estático",
    "Canal 1: Itens no Chão", 
    "Canal 2: Próprio Agente",
    "Canal 3: Outros Agentes",
    "Canal 4: Delivery Points",
    "Canal 5: Cuts Done",
    "Canal 6: Cuts Needed",
    "Canal 7: Is Ready"
]

for canal_idx, impact in canal_impacts:
    usado = "✅ USADO" if impact > 0.001 else "⚠️  Impacto baixo"
    print(f"   {canal_names[canal_idx]:30s}: {impact:.6f} {usado}")

total_usando = sum(1 for _, impact in canal_impacts if impact > 0.001)
print(f"\n🎯 RESULTADO: {total_usando}/8 canais têm impacto detectável")

if total_usando == 8:
    print("   ✅ TODOS os canais estão sendo processados pela CNN!")
else:
    print(f"   ⚠️  Apenas {total_usando} canais têm impacto significativo")
    print("   (Nota: Impacto baixo pode ser normal no início do treinamento)")

# Cleanup
test_wrapper.close()
del test_agent, test_env, test_wrapper

print("\n" + "="*70)

In [ ]:
# Função para detectar device corretamente
def get_best_device():
    """Detecta automaticamente o melhor device disponível."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        print(f"NVIDIA GPU detectada: {gpu_name}")
        print("   -> Usando CUDA (recomendado)")
        return device, f"CUDA ({gpu_name})"
    
    if "cuda" not in torch.__version__ and torch.__version__.endswith("+cpu"):
        print("PyTorch CPU detectado!")
        print("   -> Sua GPU pode usar CUDA")
        print("   -> Reinstale PyTorch CUDA para melhor performance:")
        print("   -> pip install torch --index-url https://download.pytorch.org/whl/cu118")
        print("   -> Por enquanto, usando CPU...")
        device = torch.device("cpu")
        return device, "CPU (PyTorch CPU version)"
    
    if DIRECTML_AVAILABLE:
        print("DirectML disponível")
        print("   -> Detectado como AMD GPU")
        print("   -> Usando DirectML")
        device = torch_directml.device()
        return device, "DirectML (AMD GPU)"
    
    # Fallback: CPU
    device = torch.device("cpu")
    print("💻 Usando CPU")
    print("   -> GPU não disponível ou não compatível")
    return device, "CPU"

device, device_name = get_best_device()

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
elif device.type == "cpu":
    torch.set_num_threads(4)

print(f"Device: {device_name}")

In [ ]:
def debug_agent_learning(agent, env, device, is_final=False):
    """
    Verifica se o agente está aprendendo corretamente.
    Detecta problemas como gradientes vanishing/exploding.
    
    Args:
        agent: Agente treinado
        env: Ambiente
        device: Device PyTorch
        is_final: Se True, exibe informações detalhadas
    """
    if not is_final:
        return None
    
    # 1. Testar se o agente distingue estados diferentes
    obs, _ = env.reset()
    obs_list = [obs]
    
    # Executa algumas ações para obter estados diferentes
    for _ in range(5):
        action = env.action_space.sample()
        next_obs, _, done, truncated, _ = env.step(action)
        if not (done or truncated):
            obs_list.append(next_obs)
        else:
            break
    
    diagnostics = {}
    
    # Comparar primeiro e último estado (devem ser diferentes)
    if len(obs_list) >= 2:
        obs1_tensor = torch.Tensor(obs_list[0]).unsqueeze(0).to(device)
        obs2_tensor = torch.Tensor(obs_list[-1]).unsqueeze(0).to(device)
        
        with torch.no_grad():
            _, _, _, value1 = agent.get_action_and_value(obs1_tensor)
            _, _, _, value2 = agent.get_action_and_value(obs2_tensor)
        
        value_diff = abs(value1.item() - value2.item())
        obs_diff = np.abs(obs_list[0] - obs_list[-1]).sum()
        
        diagnostics['obs_diff'] = obs_diff
        diagnostics['value_diff'] = value_diff
        diagnostics['value1'] = value1.item()
        diagnostics['value2'] = value2.item()
    
    # 2. Verificar distribuição de probabilidades de ação
    obs_tensor = torch.Tensor(obs_list[0]).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits, _ = agent.forward(obs_tensor)
        probs = torch.softmax(logits, dim=-1)
        entropy = Categorical(probs=probs).entropy()
    
    max_prob = probs.max().item()
    diagnostics['max_prob'] = max_prob
    diagnostics['entropy'] = entropy.item()
    diagnostics['action_probs'] = probs[0].cpu().numpy()
    
    return diagnostics

def format_final_diagnostics(diagnostics, action_distribution, reward_distribution):
    """Formata diagnósticos para exibição final."""
    if diagnostics is None:
        return ""
    
    output = []
    output.append("\n📊 DIAGNÓSTICO FINAL:")
    
    # Estado da rede
    output.append("\n   Estado da Rede:")
    if 'value_diff' in diagnostics:
        value_diff = diagnostics['value_diff']
        if value_diff < 0.01:
            output.append(f"      ⚠️  Valores muito similares ({value_diff:.4f}) - pode não estar aprendendo")
        else:
            output.append(f"      ✅ Diferenciação de estados: {value_diff:.4f}")
    
    # Policy
    if 'max_prob' in diagnostics:
        max_prob = diagnostics['max_prob']
        if max_prob > 0.95:
            output.append(f"      ⚠️  Muito determinístico ({max_prob:.3f}) - baixa exploração")
        elif max_prob < 0.25:
            output.append(f"      ⚠️  Muito aleatório ({max_prob:.3f}) - não aprendeu")
        else:
            output.append(f"      ✅ Exploração adequada (max_prob: {max_prob:.3f})")
        
        output.append(f"      📈 Entropia: {diagnostics['entropy']:.3f}")
    
    # Distribuição de ações durante treinamento
    total_actions = sum(action_distribution.values())
    if total_actions > 0:
        output.append("\n   Distribuição de Ações (Treinamento):")
        action_names = ['↑Up', '↓Down', '←Left', '→Right', '🔧Interact']
        
        for i, name in enumerate(action_names):
            pct = (action_distribution.get(i, 0) / total_actions) * 100
            output.append(f"      {name:12s}: {pct:5.1f}%")
        
        # Alertas
        interact_pct = (action_distribution.get(4, 0) / total_actions) * 100
        if interact_pct > 70:
            output.append(f"      ⚠️  Muito Interact - pode estar preso")
        elif interact_pct < 5:
            output.append(f"      ⚠️  Pouco Interact - não usa objetos")
    
    # Distribuição de recompensas
    total_rewards = sum(reward_distribution.values())
    if total_rewards > 0:
        output.append("\n   Distribuição de Recompensas:")
        pos_pct = (reward_distribution.get('positive', 0) / total_rewards) * 100
        neg_pct = (reward_distribution.get('negative', 0) / total_rewards) * 100
        zero_pct = (reward_distribution.get('zero', 0) / total_rewards) * 100
        
        output.append(f"      Positivas: {pos_pct:5.1f}%")
        output.append(f"      Negativas: {neg_pct:5.1f}%")
        output.append(f"      Neutras:   {zero_pct:5.1f}%")
        
        if pos_pct < 30:
            output.append(f"      ⚠️  Baixa taxa de recompensas positivas")
    
    return "\n".join(output)

def check_gradient_health(agent):
    """Verifica se os gradientes estão em uma faixa saudável."""
    total_norm = 0
    for p in agent.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    total_norm = total_norm ** 0.5
    
    return total_norm

print("✅ Funções de debugging carregadas (exibição no final)")

## 🎓 Funções de Debugging e Monitoramento

In [ ]:
results = {}

best_ppo_avg_reward = -np.inf 
best_ppo_std_reward = 0.0 
best_ppo_rewards = [] 
best_ppo_params = {}

top_10_models = []

MODEL_SAVE_DIR = "models" 
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print("\n" + "="*70)
print(f"🚀 GRID SEARCH - {len(combinations)} CONFIGURAÇÕES")
print("="*70)

start_time_total = time.time()

# Treina o agente PPO com todas as combinações de hiperparâmetros
for i, params_original in enumerate(combinations):
    
    model_filename = "_".join([f"{k}{str(v).replace('.', 'p').replace('-', 'neg')}" for k, v in params_original.items()])
    config_name = f"config_{i+1:02d}"
    
    # Cria cópia da configuração
    config = params_original.copy()
    
    # Cria novo ambiente para esta configuração
    base_env = OvercookedMARL(
        render_mode=None,  # Sem renderização durante treinamento
        map_path=ENVIRONMENT_CONFIG['map_path'],
        num_fixed_agents=ENVIRONMENT_CONFIG['num_fixed_agents'],
        verbose=False  # Desabilita logs do ambiente
    )
    env = PettingZooToGymWrapper(base_env)
    
    # Seed para reprodutibilidade
    random.seed(config['seed'])
    np.random.seed(config['seed'])
    torch.manual_seed(config['seed'])
    
    # Cria agente e otimizador
    agent = ActorCritic(env.observation_space, env.action_space).to(device)
    optimizer = optim.Adam(agent.parameters(), lr=config['learning_rate'], eps=1e-5)
    
    # Storage para rollouts
    num_steps = config['num_steps']
    obs_storage = torch.zeros((num_steps,) + env.observation_space.shape).to(device)
    actions_storage = torch.zeros(num_steps).to(device)
    logprobs_storage = torch.zeros(num_steps).to(device)
    rewards_storage = torch.zeros(num_steps).to(device)
    dones_storage = torch.zeros(num_steps).to(device)
    values_storage = torch.zeros(num_steps).to(device)
    
    # Métricas
    episode_rewards = []
    episode_lengths = []
    current_episode_reward = 0
    current_episode_length = 0
    
    # Rastreamento de ações e recompensas
    action_counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
    reward_types = {'positive': 0, 'negative': 0, 'zero': 0}
    
    # NOVO: Rastreamento de exploração
    forced_explorations = 0
    total_actions_taken = 0
    
    # Reseta o ambiente
    obs, _ = env.reset()
    obs = torch.Tensor(obs).to(device)
    
    global_step = 0
    num_updates = config['total_timesteps'] // num_steps
    
    pbar = tqdm(range(1, num_updates + 1), desc=f"Config {i+1}/{len(combinations)}")
    
    for update in pbar:
        # ====================
        # FASE 1: Coleta de Dados (Rollout)
        # ====================
        agent.eval()
        for step in range(num_steps):
            global_step += 1
            total_actions_taken += 1
            obs_storage[step] = obs
            
            # NOVO: Calcula epsilon atual (decai linearmente)
            epsilon_start = config.get('exploration_epsilon_start', 0.0)
            epsilon_end = config.get('exploration_epsilon_end', 0.0)
            decay_steps = config.get('exploration_decay_steps', config['total_timesteps'])
            
            if global_step < decay_steps:
                epsilon = epsilon_start - (epsilon_start - epsilon_end) * (global_step / decay_steps)
            else:
                epsilon = epsilon_end
            
            # Seleciona ação com exploração forçada
            with torch.no_grad():
                action_policy, logprob, _, value = agent.get_action_and_value(obs.unsqueeze(0))
            
            # NOVO: Epsilon-greedy exploration
            if random.random() < epsilon:
                # Ação aleatória para exploração
                action = torch.tensor(env.action_space.sample())
                forced_explorations += 1
                
                # Recalcula log_prob para a ação aleatória
                logits, _ = agent.forward(obs.unsqueeze(0))
                probs = Categorical(logits=logits)
                logprob = probs.log_prob(action)
            else:
                action = action_policy
            
            actions_storage[step] = action
            logprobs_storage[step] = logprob
            values_storage[step] = value.flatten()
            
            # Rastreia ação
            action_counts[int(action.item())] += 1
            
            # Step no ambiente
            next_obs, reward, terminated, truncated, info = env.step(int(action.item()))
            done = terminated or truncated
            
            # Rastreia tipo de recompensa
            if reward > 0:
                reward_types['positive'] += 1
            elif reward < 0:
                reward_types['negative'] += 1
            else:
                reward_types['zero'] += 1
            
            rewards_storage[step] = torch.tensor(reward).to(device)
            dones_storage[step] = torch.tensor(done).to(device)
            
            obs = torch.Tensor(next_obs).to(device)
            
            # Métricas de episódio
            current_episode_reward += reward
            current_episode_length += 1
            
            if done:
                episode_rewards.append(current_episode_reward)
                episode_lengths.append(current_episode_length)
                current_episode_reward = 0
                current_episode_length = 0
                obs, _ = env.reset()
                obs = torch.Tensor(obs).to(device)
        
        # ====================
        # FASE 2: Calcula Advantages (GAE)
        # ====================
        with torch.no_grad():
            next_value = agent.get_action_and_value(obs.unsqueeze(0))[3]
            advantages = torch.zeros_like(rewards_storage).to(device)
            lastgaelam = 0
            
            for t in reversed(range(num_steps)):
                if t == num_steps - 1:
                    nextnonterminal = 1.0 - dones_storage[t]
                    nextvalues = next_value
                else:
                    nextnonterminal = 1.0 - dones_storage[t]
                    nextvalues = values_storage[t + 1]
                
                delta = rewards_storage[t] + config['gamma'] * nextvalues * nextnonterminal - values_storage[t]
                advantages[t] = lastgaelam = delta + config['gamma'] * config['gae_lambda'] * nextnonterminal * lastgaelam
            
            returns = advantages + values_storage
        
        # ====================
        # FASE 3: Otimização PPO
        # ====================
        agent.train()
        
        # Flatten batch (detach para evitar backward duplo)
        b_obs = obs_storage.reshape((-1,) + env.observation_space.shape)
        b_logprobs = logprobs_storage.reshape(-1).detach()
        b_actions = actions_storage.reshape(-1).detach()
        b_advantages = advantages.reshape(-1).detach()
        b_returns = returns.reshape(-1).detach()
        b_values = values_storage.reshape(-1).detach()
        
        # Normaliza advantages
        b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)
        
        # Minibatch optimization
        batch_size = num_steps
        minibatch_size = batch_size // config['num_minibatches']
        
        for epoch in range(config['update_epochs']):
            b_inds = torch.randperm(batch_size)
            
            for start in range(0, batch_size, minibatch_size):
                end = start + minibatch_size
                mb_inds = b_inds[start:end]
                
                _, newlogprob, entropy, newvalue = agent.get_action_and_value(
                    b_obs[mb_inds], b_actions.long()[mb_inds]
                )
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()
                
                mb_advantages = b_advantages[mb_inds]
                
                # Policy loss (PPO clipping)
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - config['clip_coef'], 1 + config['clip_coef'])
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()
                
                # Value loss
                newvalue = newvalue.view(-1)
                
                # Clip value predictions to prevent extreme values
                v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                v_clipped = b_values[mb_inds] + torch.clamp(
                    newvalue - b_values[mb_inds],
                    -config['clip_coef'],
                    config['clip_coef']
                )
                v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
                v_loss = 0.5 * v_loss_max.mean()
                
                # Entropy loss
                entropy_loss = entropy.mean()
                
                # Total loss
                loss = pg_loss - config['ent_coef'] * entropy_loss + v_loss * config['vf_coef']
                
                # Check for NaN in loss
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"\n⚠️  NaN/Inf detected in loss! Skipping minibatch.")
                    print(f"   pg_loss: {pg_loss.item()}, v_loss: {v_loss.item()}, entropy: {entropy_loss.item()}")
                    continue
                
                # Backward
                optimizer.zero_grad()
                loss.backward()
                
                # Check for NaN in gradients
                grad_norm = nn.utils.clip_grad_norm_(agent.parameters(), config['max_grad_norm'])
                
                # Skip update if NaN detected
                if torch.isnan(grad_norm) or torch.isinf(grad_norm):
                    print(f"\n⚠️  NaN/Inf detected in gradients! Skipping update.")
                    optimizer.zero_grad()
                    continue
                
                optimizer.step()
        
        # Update progress bar
        if len(episode_rewards) > 0:
            recent_reward = np.mean(episode_rewards[-10:]) if len(episode_rewards) >= 10 else np.mean(episode_rewards)
            exploration_rate = (forced_explorations / total_actions_taken) * 100
            pbar.set_postfix({
                'reward': f"{recent_reward:.1f}",
                'episodes': len(episode_rewards),
                'interact%': f"{(action_counts[4]/sum(action_counts.values()))*100:.0f}",
                'explore%': f"{exploration_rate:.0f}"
            })
    
    # ====================
    # DIAGNÓSTICO FINAL (após completar treinamento)
    # ====================
    diagnostics = debug_agent_learning(agent, env, device, is_final=True)
    
    # Fecha o ambiente
    env.close()
    
    # Calcula recompensa média final
    avg_reward_last_100 = np.mean(episode_rewards[-100:]) if len(episode_rewards) >= 100 else np.mean(episode_rewards)
    
    # Métricas detalhadas
    total_actions = sum(action_counts.values())
    interact_percentage = (action_counts[4] / total_actions * 100) if total_actions > 0 else 0
    exploration_percentage = (forced_explorations / total_actions_taken * 100) if total_actions_taken > 0 else 0
    
    # Salva resultados
    results[config_name] = {
        'params': params_original.copy(),
        'rewards': episode_rewards.copy(),
        'lengths': episode_lengths.copy(),
        'avg_reward': avg_reward_last_100,
        'action_distribution': {k: v/total_actions for k, v in action_counts.items()},
        'interact_percentage': interact_percentage,
        'reward_distribution': reward_types,
        'exploration_percentage': exploration_percentage
    }
    
    # Gerencia top 10 modelos
    if len(top_10_models) < 10:
        top_10_models.append((avg_reward_last_100, params_original.copy(), episode_rewards.copy()))
        top_10_models.sort(key=lambda x: x[0], reverse=True)
    else:
        if avg_reward_last_100 > top_10_models[-1][0]:
            top_10_models.pop()
            top_10_models.append((avg_reward_last_100, params_original.copy(), episode_rewards.copy()))
            top_10_models.sort(key=lambda x: x[0], reverse=True)
    
    # Salva modelo com parâmetros no nome
    params_str = (
        f"lr{config['learning_rate']:.0e}_"
        f"steps{config['num_steps']}_"
        f"epochs{config['update_epochs']}_"
        f"ent{config['ent_coef']:.2f}_"
        f"clip{config['clip_coef']}_"
        f"mb{config['num_minibatches']}_"
        f"g{config['gamma']}_"
        f"lam{config['gae_lambda']}"
    )
    model_path = os.path.join(MODEL_SAVE_DIR, f"{config_name}_{params_str}_reward_{avg_reward_last_100:.2f}.pth")
    
    torch.save({
        'model_state_dict': agent.state_dict(),
        'config': config,
        'reward': avg_reward_last_100,
        'episode_rewards': episode_rewards,
        'episode_lengths': episode_lengths,
        'action_distribution': action_counts,
        'reward_distribution': reward_types
    }, model_path)
    
    # Atualiza melhor modelo
    if avg_reward_last_100 > best_ppo_avg_reward:
        best_ppo_avg_reward = avg_reward_last_100
        best_ppo_params = params_original.copy()
        best_ppo_std_reward = np.std(episode_rewards[-100:]) if len(episode_rewards) >= 100 else np.std(episode_rewards)
        best_ppo_rewards = episode_rewards.copy()
    
    # ====================
    # EXIBE RESUMO FINAL DO MODELO
    # ====================
    print(f"\n{'='*70}")
    print(f"✅ Config {i+1}/{len(combinations)} concluída")
    print(f"{'='*70}")
    print(f"   Reward médio: {avg_reward_last_100:.2f}")
    print(f"   Episódios:    {len(episode_rewards)}")
    print(f"   Interact%:    {interact_percentage:.1f}%")
    print(f"   Exploração forçada: {exploration_percentage:.1f}% das ações")
    
    # Exibe diagnósticos
    if diagnostics:
        print(format_final_diagnostics(diagnostics, action_counts, reward_types))
    
    print(f"{'='*70}\n")

training_time_total = time.time() - start_time_total

print("\n" + "="*70)
print("🎉 TREINAMENTO CONCLUÍDO")
print("="*70)
print(f"⏱️  Tempo total: {training_time_total/60:.1f} min ({training_time_total/3600:.2f}h)")
print(f"📊 Configurações testadas: {len(combinations)}")
print(f"🏆 Melhor reward: {best_ppo_avg_reward:.2f} ± {best_ppo_std_reward:.2f}")
print(f"📁 Modelos salvos em: {MODEL_SAVE_DIR}")
print(f"\n💡 Próximo passo: Execute o notebook 'Avaliacao_Modelos.ipynb' para análise detalhada")